In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
Data = [
    (1, 'sai', 'hyd', 100),
    (2, 'ram', 'hyd', 200),
    (3, 'raju', 'hyd', 300),
    (4, 'ramesh', 'delhi', 400),
    (5, 'surech', 'chennai', 500)
]

schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("Employe_name", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Salary", IntegerType(), True)
])

DF = spark.createDataFrame(Data, schema = schema)

## Narrow Transformations

Narrow transformations in Spark are operations where each partition of the parent RDD/DataFrame is used by at most one partition of the child. These are efficient and do not require shuffling data across the cluster.

**Common Narrow Transformations **

- **map**: Applies a function to each element in the dataset, returning a new dataset of the same length.
  - `DF.withColumn("Salary_plus_10", DF.Salary + 10).show()`
- **filter**: Returns a new dataset containing only elements that satisfy a given condition.
  - `DF.filter(DF.Salary > 200).show()`
- **select**: Chooses specific columns from a DataFrame.
  - `DF.select("Employe_name", "City").show()`
- **flatMap**: Similar to `map`, but can return zero or more output items for each input item.
  - `DF.select(explode(split(DF.Employe_name, " ")).alias("NamePart")).show()` *(if names had spaces)*
- **union**: Combines two datasets into one, preserving all elements.
  - `DF.union(DF).show()`
- **sample**: Returns a random sample of the dataset.
  - `DF.sample(False, 0.5).show()`
- **mapPartitions**: Applies a function to each partition of the dataset.
  - `DF.mapInPandas(lambda df: df.assign(Salary2=df.Salary*2), schema=DF.schema).show()`
- **glom**: Groups all elements within a partition into a list.
  - `DF.rdd.glom().toDF().show()` *(glom is available via RDD, but can be converted back to DataFrame)*

Narrow transformations are preferred for performance as they avoid expensive data shuffles.

In [0]:
from pyspark.sql.functions import col
DF = DF.filter(col('ID') == 2)
DF.printSchema()
DF.show()

In [0]:
DF.explain()

## Wide Transformation

Wide transformations in Spark are operations where data from multiple partitions must be shuffled across the cluster. These transformations require data movement and are generally more expensive than narrow transformations.

**Common Wide Transformations:**

- **groupBy**: Groups rows based on column values.
  
  DF.groupBy("City").count().show()
  
- **join**: Combines two DataFrames based on a key.

  DF.join(DF, DF.ID == DF.ID, "inner").show()
  
- **distinct**: Removes duplicate rows.

  DF.distinct().show()
  
- **repartition**: Changes the number of partitions, causing a shuffle.

  DF.repartition(3).show()
  
- **coalesce**: Reduces the number of partitions, may cause shuffle if increasing partitions.

  DF.coalesce(1).show()
  
- **aggregate**: Performs aggregation operations.

  DF.groupBy("City").agg(sum("Salary")).show()
  
- **sort**: Orders rows, may require shuffling.

  DF.sort("Salary").show()
  

Wide transformations are less efficient due to shuffling, so use them judiciously.

In [0]:
Data = [
    (1, 'sai', 'hyd', 100),
    (2, 'ram', 'hyd', 200),
    (3, 'raju', 'hyd', 300),
    (4, 'ramesh', 'delhi', 400),
    (5, 'surech', 'chennai', 500)
]

schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("Employe_name", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Salary", IntegerType(), True)
])

DF2 = spark.createDataFrame(Data, schema = schema)

In [0]:
DF2.columns

In [0]:
DF3 = DF2.groupBy(col("City")).agg(avg(col("salary")).alias("avg_salary"))

In [0]:
DF3.columns

In [0]:
display(DF3)

In [0]:
DF3.explain()